# General Information

EMG Signal for gesture recognition

https://www.kaggle.com/datasets/sojanprajapati/emg-signal-for-gesture-recognition/data

Information below taken from kaggle page.

So there are 11 columns. This is a Readme file from the actual dataset.https://archive.ics.uci.edu/ml/datasets/EMG+data+for+gestures

For recording patterns, we used a MYO Thalmic bracelet worn on a user’s forearm, and a PC with a Bluetooth receiver. The bracelet is equipped with eight sensors equally spaced around the forearm that simultaneously acquire myographic signals. The signals are sent through a Bluetooth interface to a PC.
We present raw EMG data for 36 subjects while they performed series of static hand gestures.The subject performs two series, each of which consists of six (seven) basic gestures. Each gesture was performed for 3 seconds with a pause of 3 seconds between gestures.

Description of raw_data _*** file
Each file consist of 10 columns:
1) Time - time in ms;
2-9) Channel - eight EMG channels of MYO Thalmic bracelet;
10) Class –the label of gestures:
0 - unmarked data,
1 - hand at rest,
2 - hand clenched in a fist,
3 - wrist flexion,
4 – wrist extension,
5 – radial deviations,
6 - ulnar deviations,
7 - extended palm (the gesture was not performed by all subjects).

Along with this, I have just added "label" column that refers to the subject who performed the experiment. There were 36 subjects, each performed 7 gestures twice.

**Enabling GPU runtime (necessary)**

Before running this notebook, click edit in the top left, then click notebook settings. Click T4 GPU, then click save. This will enable a GPU runtime, which is needed for running our neural networks

Alternatively, you can click the drop down next to your session resources (located in the top right). Click change runtime time and select T4 GPU.

# Imports and configurations

In [1]:
import numpy as np
import pandas as pd
import torch
import scipy
from torch.utils.data import TensorDataset as TData
from torch.utils.data import DataLoader as DL
import matplotlib.pyplot as plt

In [2]:
# checks if cuda is available, then sets device to GPU
if torch.cuda.is_available():
  device = 'cuda'
else:
  device = 'cpu'
device # should say cuda if gpu is enabled

'cuda'

In [ ]:
# uncomment line below for GPU info
# !nvidia-smi

Get the dataset into colab environment


In [3]:
from google.colab import files
uploaded = files.upload()

Saving EMG-data.csv to EMG-data.csv


In [4]:
dataset = pd.read_csv("EMG-data.csv") # loading in the csv dataset with pandas

# Exploratory Data Analysis and Preprocessing

In [5]:
dataset.head()

,time,channel1,channel2,channel3,channel4,channel5,channel6,channel7,channel8,class,label
0,1,0.00001,-0.00002,-0.00001,-0.00003,0.00000,-0.00001,0.00000,-0.00001,0,1
1,5,0.00001,-0.00002,-0.00001,-0.00003,0.00000,-0.00001,0.00000,-0.00001,0,1
2,6,-0.00001,0.00001,0.00002,0.00000,0.00001,-0.00002,-0.00001,0.00001,0,1
3,7,-0.00001,0.00001,0.00002,0.00000,0.00001,-0.00002,-0.00001,0.00001,0,1
4,8,-0.00001,0.00001,0.00002,0.00000,0.00001,-0.00002,-0.00001,0.00001,0,1


In [6]:
#only interested in channels and class values that are 1-6 because 0 is unmarked data and not all subjects performed 7
#IMPORTANT: based on previous emg_optuna confusion matrix results, dropping gestures 2,3,6 and only predicting 1,4,5

# Keep only rows where class is 1,4,5
filtered_emg = dataset.loc[dataset['class'].isin([1, 4, 5])]

# Drop unnecessary columns: time
filtered_emg = filtered_emg.drop(columns=['time'])

# Check the result
print(filtered_emg['class'].unique())  # Should show [1 4 5]


[1 4 5]


In [7]:
filtered_emg.head() #should only have the channels, class and label

,channel1,channel2,channel3,channel4,channel5,channel6,channel7,channel8,class,label
2287,-0.00001,0.00000,-0.00001,0.00000,0.00000,-0.00001,-0.00001,0.00001,1,1
2288,-0.00001,-0.00002,0.00000,-0.00001,-0.00001,-0.00001,-0.00003,-0.00002,1,1
2289,-0.00001,-0.00002,0.00000,-0.00001,-0.00001,-0.00001,-0.00003,-0.00002,1,1
2290,-0.00001,-0.00002,0.00000,-0.00001,-0.00001,-0.00001,-0.00003,-0.00002,1,1
2291,-0.00001,-0.00002,0.00000,-0.00001,-0.00001,-0.00001,-0.00003,-0.00002,1,1


In [8]:
#If a subject does gesture 1 twice, you get two separate arrays instead of one big merged array.
#This preserves the temporal structure of each repetition — very important for EMG and LSTMs.
#Separates contiguous sequences of the same class.
#So if a subject performs a gesture twice, you get two separate arrays instead of one big concatenation.
#This is often important for EMG, because signals drift over time, and concatenating two separate repetitions can confuse models (especially LSTMs that look at sequences).

# Get all unique subject IDs from the dataset.
# filtered_emg['label'] holds the subject identifier for each row.
subjects = filtered_emg['label'].unique()   # label = subject ID


# Dictionary to store each subject's EMG blocks.
# Structure:
#   {
#     subject_id: [(class, array), (class, array), ...],
#     ...
#   }
subject_signals = {} #list of signals for each subject in the form subject_signals = {1: [(class, arr), (class, arr), ...], 2: [(class, arr), (class, arr)], ...}

# Process data for each subject individually
for subj in subjects:
    # Extract only the rows corresponding to this subject
    df = filtered_emg[filtered_emg['label'] == subj].copy()  # filter by subject ID

    # create adjacent blocks whenever class changes
    df['block'] = (df['class'] != df['class'].shift()).cumsum()

    # List to store (class, EMG array) pairs for this subject
    subj_blocks = []

    # Iterate through each adjacent gesture block
    for block_id, block_df in df.groupby('block'):

      # Get the gesture class for this block
        cls = block_df['class'].iloc[0]

      # We only keep gesture classes 1,4,5 (ignore rest/noise/other labels)
        if cls in [1,4,5]:  # valid gesture classes only

            # Extract the 8 EMG channels.
            # block_df.iloc[:, 1:9] means columns 1–8 (assuming EMG channels start at column 1).
            # Convert to numpy and transpose → shape = (8 channels, T samples)
            arr = block_df.iloc[:, 1:9].to_numpy().T  # 8 × T

            # Save the (class, EMG block) tuple
            subj_blocks.append((cls, arr))

    # Store all gesture blocks for this subject
    subject_signals[subj] = subj_blocks


In [9]:
class_signals = {1:[], 4:[], 5:[]} #make lists for each class

In [10]:
# Create a dictionary where each gesture class maps to an empty list.
# After filling, each class will contain all EMG arrays belonging to that gesture.
class_signals = {c: [] for c in [1, 4, 5]}

# Iterate through each subject and their list of (class, EMG array) blocks
for subj, blocks in subject_signals.items():
    for class_label, arr in blocks:
        # Append the EMG block to the corresponding class list
        class_signals[class_label].append(arr)

# After this:
#   class_signals[1] contains ALL EMG arrays labeled as gesture 1 (across all subjects)
#   class_signals[4] contains ALL gesture 4 arrays
#   ...
#   class_signals[5] contains ALL gesture 5 arrays

In [11]:
# Lists to hold (class, signal) pairs for each split
train = []
val = []
test = []

# Fraction of each signal reserved for validation + test.
# Same ratio used in the original sample pipeline.
eval_portion = 0.4  # same as sample pipeline

# Loop over each gesture class
for c in [1, 4, 5]:
    signals = class_signals[c] # all EMG arrays for this class

    # Process each individual signal sequence
    for sig in signals:
        seq_len = sig.shape[1] # number of time steps (T)
        eval_len = int(eval_portion * seq_len)  # size of eval window

        # Randomly choose a window inside the signal to hold out for eval/test
        # start ∈ [0, seq_len - eval_len)
        start = np.random.randint(0, seq_len - eval_len)
        end = start + eval_len # end index of the eval window

        # training: remove eval window
        train.append((c, sig[:, list(range(start)) + list(range(end, seq_len))]))

        # validation: first half of eval window
        val.append((c, sig[:, start:start+eval_len//2]))

        # test: second half
        test.append((c, sig[:, start+eval_len//2:end]))

# Signal Preprocessing

In [12]:
#bandpass filter
def bandpass_filter(signal, crit_freq = [10, 60], sampling_freq = 125):

  # Order of the Butterworth filter (higher = sharper cutoff)
  order = 4

  # Design a Butterworth bandpass filter with critical frequencies (5–40 Hz)
  # fs specifies sampling frequency, so scipy automatically normalizes cutoff
  b, a = scipy.signal.butter(order, crit_freq, btype = 'bandpass', fs = sampling_freq)

  # Apply zero-phase filtering using filtfilt (avoids phase distortion)
  # axis=1 because signals have shape: channels × time
  processed_signal = scipy.signal.filtfilt(b, a, signal, axis = 1)
  return processed_signal

#segmentation
def segmentation(signal, sampling_freq=125, window_size=1, window_shift=0.05):

  # Convert window size (seconds) → number of samples
  w_size = int(sampling_freq * window_size)

  # Convert shift (seconds) → hop length in samples
  w_shift = int(sampling_freq * window_shift)

  segments = [] # List to store the segmented windows
  i = 0 # Sliding window starting index

  # Slide window until the last full window fits in the signal
  while i + w_size <= signal.shape[1]:
    # Extract a window of shape: channels × w_size
    segments.append(signal[:, i: i + w_size])

    # Move the window forward by the shift amount
    i += w_shift
  return segments

#full preprocess method
def preprocess_emg(signal, fs=125, band=[5,40], window_size=1, window_shift=0.05):

    # 1. Bandpass filtering to remove noise and keep muscle-relevant EMG frequencies
    filtered = bandpass_filter(signal, band, fs)

    # 2. Channel-wise z-score normalization:
    #  subtract mean and divide by std for each channel independently
    normed = (filtered - filtered.mean(1, keepdims=True)) / filtered.std(1, keepdims=True)

    # 3. Segment the normalized signal into overlapping windows
    preprocessed = segmentation(normed, fs, window_size, window_shift)
    return preprocessed

improved preprocessing

In [13]:
from sklearn.utils.class_weight import compute_class_weight
import scipy.signal
import numpy as np

def create_more_training_data(dataset, fs=125, band=[5, 40]):
    """
    Create more training samples using smaller window shift
    This is the easiest way to boost accuracy
    """
    processed = []

    # Use smaller shift for MORE OVERLAPPING WINDOWS = MORE DATA
    window_size = 1.0
    window_shift = 0.05  # Changed from 0.1 to 0.05 = 2x more data!

    for label, sig in dataset:
        # Bandpass filter
        order = 4
        b, a = scipy.signal.butter(order, band, btype='bandpass', fs=fs)
        filtered = scipy.signal.filtfilt(b, a, sig, axis=1)

        # Normalize
        normed = (filtered - filtered.mean(1, keepdims=True)) / (filtered.std(1, keepdims=True) + 1e-8)

        # Segment with smaller shift
        w_size = int(fs * window_size)
        w_shift = int(fs * window_shift)

        i = 0
        while i + w_size <= sig.shape[1]:
            segments = normed[:, i:i + w_size]
            processed.append((label, segments))
            i += w_shift

    return processed

In [14]:
# function to preprocess a dataset of (label, signal) tuples
def preprocess_dataset(dataset, fs=125, band=[5,40], window_size=1, window_shift=0.05):
    processed = []
    for label, sig in dataset:
        segments = preprocess_emg(sig, fs=fs, band=band, window_size=window_size, window_shift=window_shift)
        # attach label to each segment
        processed.extend([(label, seg) for seg in segments])
    return processed

# apply to train, val, test
# train_processed = preprocess_dataset(train)
# val_processed   = preprocess_dataset(val)
# test_processed  = preprocess_dataset(test)
train_processed = preprocess_dataset(train, fs=125, band=[5,40], window_size=1, window_shift=0.05)
val_processed   = preprocess_dataset(val, fs=125, band=[5,40], window_size=1, window_shift=0.05)
test_processed  = preprocess_dataset(test, fs=125, band=[5,40], window_size=1, window_shift=0.05)

# check results
print(f"Train segments: {len(train_processed)}, Val segments: {len(val_processed)}, Test segments: {len(test_processed)}")
print(f"Segment shape: {train_processed[0][1].shape}")  # [1] gives the NumPy array


Train segments: 66635, Val segments: 16328, Test segments: 16354
Segment shape: (8, 125)


# Creating Data Loaders

In [15]:
import torch
from torch.utils.data import TensorDataset as TData, DataLoader as DL

num_classes = 3  # gestures 1,4,5
label_map = {1: 0, 4: 1, 5: 2}

# create torch tensor of zeros to hold data
train_emg_tensor = torch.zeros((len(train_processed), train_processed[0][1].shape[0], train_processed[0][1].shape[1]))
val_emg_tensor   = torch.zeros((len(val_processed), val_processed[0][1].shape[0], val_processed[0][1].shape[1]))
test_emg_tensor  = torch.zeros((len(test_processed), test_processed[0][1].shape[0], test_processed[0][1].shape[1]))

# add each sample in train, validation, and test lists to appropriate tensor at correct index
for i, (label, seg) in enumerate(train_processed):
    train_emg_tensor[i] = torch.from_numpy(seg.copy())
for i, (label, seg) in enumerate(val_processed):
    val_emg_tensor[i] = torch.from_numpy(seg.copy())
for i, (label, seg) in enumerate(test_processed):
    test_emg_tensor[i] = torch.from_numpy(seg.copy())

# create zero tensor for one hot encoded labels
train_label_tensor = torch.zeros(len(train_processed), num_classes)
val_label_tensor   = torch.zeros(len(val_processed), num_classes)
test_label_tensor  = torch.zeros(len(test_processed), num_classes)

# add labels to tensor at correct index
for i, (label, seg) in enumerate(train_processed):
    train_label_tensor[i][label_map[label]] = 1
for i, (label, seg) in enumerate(val_processed):
    val_label_tensor[i][label_map[label]] = 1
for i, (label, seg) in enumerate(test_processed):
    test_label_tensor[i][label_map[label]] = 1

# convert input, target tensors to Tensor Dataset from torch
train_ds = TData(train_emg_tensor, train_label_tensor)
val_ds   = TData(val_emg_tensor, val_label_tensor)
test_ds  = TData(test_emg_tensor, test_label_tensor)
# create dataloaders to hold batched data (batch size chosen was 64)
batch_size = 128
train_dl = DL(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_dl   = DL(val_ds, batch_size=batch_size, shuffle=True, drop_last=True)
test_dl  = DL(test_ds, batch_size=batch_size, shuffle=True, drop_last=True)

#check
print(f'Train batches: {len(train_dl)}, Val batches: {len(val_dl)}, Test batches: {len(test_dl)}')
print(f'Sample shape: {train_emg_tensor[0].shape}, Label shape: {train_label_tensor[0].shape}')


Train batches: 520, Val batches: 127, Test batches: 127
Sample shape: torch.Size([8, 125]), Label shape: torch.Size([3])


# Improved Model and Optuna


In [16]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 34.3 MB/s eta 0:00:00


In [17]:
import optuna
import torch
import torch.nn as nn
from tqdm import tqdm
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix

Data Augmentation

In [18]:
class EMGAugmentation:
    """Advanced augmentation for EMG signals"""

    @staticmethod
    def add_gaussian_noise(signal, noise_factor=0.005):
        noise = torch.randn_like(signal) * noise_factor
        return signal + noise

    @staticmethod
    def time_shift(signal, shift_range=10):
        shift = np.random.randint(-shift_range, shift_range)
        return torch.roll(signal, shift, dims=-1)

    @staticmethod
    def amplitude_scale(signal, scale_range=(0.9, 1.1)):
        scale = np.random.uniform(*scale_range)
        return signal * scale

    @staticmethod
    def channel_dropout(signal, drop_prob=0.1):
        batch_size, channels, time_steps = signal.shape
        mask = (torch.rand(batch_size, channels, 1) > drop_prob).float().to(signal.device)
        return signal * mask

    @staticmethod
    def augment(signal, prob=0.8):
        """Apply random combination of augmentations during training"""
        if np.random.rand() > prob:
            return signal

        augmentations = []
        if np.random.rand() > 0.5:
            augmentations.append(lambda x: EMGAugmentation.add_gaussian_noise(x, 0.005))
        if np.random.rand() > 0.5:
            augmentations.append(lambda x: EMGAugmentation.time_shift(x, 10))
        if np.random.rand() > 0.5:
            augmentations.append(lambda x: EMGAugmentation.amplitude_scale(x, (0.95, 1.05)))
        if np.random.rand() > 0.7:
            augmentations.append(lambda x: EMGAugmentation.channel_dropout(x, 0.1))

        for aug in augmentations:
            signal = aug(signal)

        return signal

New Model Components

In [19]:
class SqueezeExcitation(nn.Module):
    """Channel attention mechanism"""
    def __init__(self, channels, reduction=8):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        batch, channels, time = x.shape
        squeeze = x.mean(dim=2)
        excitation = F.relu(self.fc1(squeeze))
        excitation = torch.sigmoid(self.fc2(excitation))
        excitation = excitation.unsqueeze(2)
        return x * excitation


class MultiScaleConv1D(nn.Module):
    """Multi-scale temporal convolution"""
    def __init__(self, in_channels, out_channels, kernels=[3, 5, 7]):
        super().__init__()

        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_channels, out_channels // len(kernels),
                         kernel_size=k, padding='same'),
                nn.BatchNorm1d(out_channels // len(kernels)),
                nn.ReLU()
            ) for k in kernels
        ])

        self.out_channels = (out_channels // len(kernels)) * len(kernels)

    def forward(self, x):
        outputs = [conv(x) for conv in self.convs]
        return torch.cat(outputs, dim=1)


class AttentionLSTM(nn.Module):
    """LSTM with self-attention"""
    def __init__(self, input_size, hidden_size, num_layers=1, bidirectional=True, dropout=0.2):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=bidirectional,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        lstm_out_size = hidden_size * 2 if bidirectional else hidden_size

        # Attention
        self.attention = nn.Sequential(
            nn.Linear(lstm_out_size, lstm_out_size // 2),
            nn.Tanh(),
            nn.Linear(lstm_out_size // 2, 1)
        )

        self.output_size = lstm_out_size

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attention_weights = F.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attention_weights * lstm_out, dim=1)
        return context

Improved Model

In [20]:
class ImprovedCNNLSTMModel(nn.Module):
    def __init__(self, in_channels, n_classes, trial):
        super().__init__()

        conv1_filters = trial.suggest_categorical('conv1_filters', [96, 128])
        self.multiscale_conv1 = MultiScaleConv1D(in_channels, conv1_filters, kernels=[3, 5, 7, 9])
        actual_conv1_filters = self.multiscale_conv1.out_channels

        pool1_size = trial.suggest_int('pool1_size', 2, 3)
        self.pool1 = nn.MaxPool1d(pool1_size)
        self.se1 = SqueezeExcitation(actual_conv1_filters, reduction=8)

        sep_filters = trial.suggest_categorical('sep_filters', [128, 192])
        self.multiscale_conv2 = MultiScaleConv1D(actual_conv1_filters, sep_filters, kernels=[3, 5, 7])
        actual_sep_filters = self.multiscale_conv2.out_channels

        pool2_size = trial.suggest_int('pool2_size', 2, 3)
        self.pool2 = nn.MaxPool1d(pool2_size)
        self.se2 = SqueezeExcitation(actual_sep_filters, reduction=8)

        conv3_filters = trial.suggest_categorical('conv3_filters', [128, 192])
        self.conv3 = nn.Sequential(
            nn.Conv1d(actual_sep_filters, conv3_filters, kernel_size=3, padding='same'),
            nn.BatchNorm1d(conv3_filters),
            nn.ReLU(),
            nn.MaxPool1d(2)
        )

        #lstm_hidden = trial.suggest_categorical('lstm_hidden', [128, 256, 512])
        lstm_hidden = 512
        #lstm_layers = trial.suggest_int('lstm_layers', 1, 2)
        lstm_layers = 2
        lstm_dropout = trial.suggest_float('lstm_dropout', 0.3, 0.5) if lstm_layers > 1 else 0.3

        self.attention_lstm = AttentionLSTM(
            input_size=in_channels,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            bidirectional=True,
            dropout=lstm_dropout
        )
        lstm_out_size = self.attention_lstm.output_size

        # FIX: dummy pass to compute CNN output size so FC layers register properly
        dummy = torch.zeros(1, in_channels, 125)
        with torch.no_grad():
            out = self.multiscale_conv1(dummy)
            out = self.pool1(out); out = self.se1(out)
            out = self.multiscale_conv2(out)
            out = self.pool2(out); out = self.se2(out)
            out = self.conv3(out)
            cnn_out_size = out.view(1, -1).shape[1]

        fusion_size = cnn_out_size + lstm_out_size

        # FIX: lowered dropout range from (0.5, 0.7) to (0.3, 0.5) to this
        fc_dropout = trial.suggest_float('fc_dropout', 0.45, 0.6)
        self.fusion_dropout = nn.Dropout(fc_dropout)

        # FIX: FC layers defined in __init__ so optimizer can actually train them
        self.fc1 = nn.Linear(fusion_size, 512)
        self.fc1_bn = nn.BatchNorm1d(512)
        self.fc2 = nn.Linear(512, 256)
        self.fc2_bn = nn.BatchNorm1d(256)
        self.fc_out = nn.Linear(256, n_classes)

    def forward(self, x):
        cnn_out = self.multiscale_conv1(x)
        cnn_out = self.pool1(cnn_out)
        cnn_out = self.se1(cnn_out)

        cnn_out = self.multiscale_conv2(cnn_out)
        cnn_out = self.pool2(cnn_out)
        cnn_out = self.se2(cnn_out)

        cnn_out = self.conv3(cnn_out)
        cnn_out = cnn_out.view(cnn_out.size(0), -1)

        lstm_input = x.permute(0, 2, 1)
        lstm_out = self.attention_lstm(lstm_input)

        fused = torch.cat([cnn_out, lstm_out], dim=1)
        out = F.relu(self.fc1_bn(self.fc1(fused)))
        out = self.fusion_dropout(out)
        out = F.relu(self.fc2_bn(self.fc2(out)))
        out = self.fusion_dropout(out)
        out = self.fc_out(out)

        return out

Label Smoothing

In [21]:
class LabelSmoothingCrossEntropy(nn.Module):
    """Label smoothing for better generalization"""
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, pred, target):
        n_classes = pred.size(1)
        smooth_target = target * (1 - self.smoothing) + self.smoothing / n_classes
        log_prob = F.log_softmax(pred, dim=1)
        loss = -(smooth_target * log_prob).sum(dim=1).mean()
        return loss


def mixup_data(x, y, alpha=0.2):
    """Mixup augmentation"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1

    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]

    return mixed_x, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

Improved Training

In [22]:
def train_and_validate_improved(trial, model, optimizer, scheduler, epochs, use_mixup=True):
    """Improved training loop"""

    criterion = LabelSmoothingCrossEntropy(smoothing=0.1)

    val_losses = []
    val_accs = []
    best_val_acc = 0.0
    patience_counter = 0
    early_stop_patience = 12

    for epoch in range(epochs):
        # Training
        model.train()
        total_train_loss = 0.0
        train_correct = 0
        train_total = 0

        for signals, labels in train_dl:
            signals = signals.to(device).to(torch.float32)
            labels = labels.to(device).to(torch.float32)

            # Augmentation
            signals = EMGAugmentation.augment(signals, prob=.5)

            # Mixup
            if use_mixup and np.random.rand() > 0.7:
                signals, labels_a, labels_b, lam = mixup_data(signals, labels, alpha=0.2)
                optimizer.zero_grad()
                outputs = model(signals)
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            else:
                optimizer.zero_grad()
                outputs = model(signals)
                loss = criterion(outputs, labels)

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_train_loss += loss.item()
            pred_classes = outputs.argmax(dim=1)
            true_classes = labels.argmax(dim=1)
            train_correct += (pred_classes == true_classes).sum().item()
            train_total += labels.size(0)

        avg_train_loss = total_train_loss / len(train_dl)
        train_acc = train_correct / train_total

        # Validation
        model.eval()
        total_val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for signals, labels in val_dl:
                signals = signals.to(device).to(torch.float32)
                labels = labels.to(device).to(torch.float32)

                outputs = model(signals)
                loss = criterion(outputs, labels)

                total_val_loss += loss.item()
                pred_classes = outputs.argmax(dim=1)
                true_classes = labels.argmax(dim=1)
                val_correct += (pred_classes == true_classes).sum().item()
                val_total += labels.size(0)

        avg_val_loss = total_val_loss / len(val_dl)
        avg_val_acc = val_correct / val_total

        val_losses.append(avg_val_loss)
        val_accs.append(avg_val_acc)

        # Scheduler
        if scheduler is not None:
            if isinstance(scheduler, torch.optim.lr_scheduler.CosineAnnealingWarmRestarts):
                scheduler.step()
            else:
                scheduler.step(avg_val_loss)

        # Save best
        if avg_val_acc > best_val_acc:
            best_val_acc = avg_val_acc
            trial.set_user_attr('tuned_model', model.state_dict())
            trial.set_user_attr('best_val_acc', best_val_acc)
            patience_counter = 0
        else:
            patience_counter += 1

        # Print
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"  Epoch {epoch+1}/{epochs} - "
                  f"Train Loss: {avg_train_loss:.4f}, Train Acc: {train_acc:.4f}, "
                  f"Val Loss: {avg_val_loss:.4f}, Val Acc: {avg_val_acc:.4f}")

        # Pruning
        trial.report(-avg_val_acc, epoch)  # negative because Optuna minimizes
        if trial.should_prune():
            raise optuna.TrialPruned()


        # Early stopping
        if patience_counter >= early_stop_patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    trial.set_user_attr('final_epoch', epoch + 1)

    return -best_val_acc  # Optuna minimizes, so negate accuracy


Improved Objective

In [23]:
def improved_objective(trial):
    print(f"\n{'='*60}")
    print(f"Trial {trial.number}")
    print(f"{'='*60}")

    model = ImprovedCNNLSTMModel(in_channels=8, n_classes=3, trial=trial)
    model.to(device)

    lr = trial.suggest_float("lr", 1e-4, 5e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    #use_cosine = trial.suggest_categorical("use_cosine_scheduler", [True, False])
    use_cosine = False
    if use_cosine:
        epochs = trial.suggest_int('epochs', 40, 60)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            optimizer, T_0=10, T_mult=2
        )
    else:
        epochs = trial.suggest_int('epochs', 40, 60)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5
        )

    #use_mixup = trial.suggest_categorical("use_mixup", [True, False])
    use_mixup = True

    avg_val_loss = train_and_validate_improved(
        trial, model, optimizer, scheduler, epochs, use_mixup
    )

    return avg_val_loss

def callback(study, trial):
    if study.best_trial.number == trial.number:
        study.set_user_attr(key="best_model_state", value=trial.user_attrs["tuned_model"])
        study.set_user_attr(key="best_val_acc", value=trial.user_attrs.get("best_val_acc", 0.0))

Test Time Augmentation

In [24]:
def test_with_tta(model, test_dl, n_tta=5):
    """Test with Test-Time Augmentation"""
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for signals, labels in tqdm(test_dl, desc="Testing with TTA"):
            signals = signals.to(device).to(torch.float32)
            labels = labels.to(device).to(torch.float32)

            # Multiple predictions with light augmentation
            tta_outputs = []
            for _ in range(n_tta):
                aug_signals = EMGAugmentation.add_gaussian_noise(signals, 0.002)
                outputs = model(aug_signals)
                tta_outputs.append(outputs)

            # Average
            avg_output = torch.stack(tta_outputs).mean(dim=0)

            pred_classes = avg_output.argmax(dim=1)
            true_classes = labels.argmax(dim=1)

            all_preds.extend(pred_classes.cpu().numpy())
            all_labels.extend(true_classes.cpu().numpy())

    return np.array(all_preds), np.array(all_labels)

Run Optimization

In [25]:
# Run optimization
base_pruner = optuna.pruners.MedianPruner(n_startup_trials=3, n_warmup_steps=5)
patient_pruner = optuna.pruners.PatientPruner(base_pruner, patience=3)

study = optuna.create_study(
    direction='minimize',
    pruner=patient_pruner,
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(improved_objective, n_trials=15, callbacks=[callback])

# Print results
print("\n" + "="*60)
print("OPTIMIZATION COMPLETE")
print("="*60)
print(f"Number of finished trials: {len(study.trials)}")
print(f"Number of pruned trials: {len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])}")
print(f"\nBest trial:")
print(f"  Best val accuracy: {-study.best_trial.value:.4f}")  # negate back to positive
print(f"\nBest hyperparameters:")
for key, value in study.best_trial.params.items():
    print(f"  {key}: {value}")

/tmp/ipython-input-4123370473.py:3: ExperimentalWarning: PatientPruner is experimental (supported from v2.8.0). The interface can change in the future.
  patient_pruner = optuna.pruners.PatientPruner(base_pruner, patience=3)
[I 2026-02-19 20:18:28,990] A new study created in memory with name: no-name-da0035af-17fd-4881-a3b6-a2dd3bbc595e



Trial 0
  Epoch 1/57 - Train Loss: 0.8905, Train Acc: 0.5787, Val Loss: 1.0181, Val Acc: 0.6031
  Epoch 5/57 - Train Loss: 0.4443, Train Acc: 0.8627, Val Loss: 0.8504, Val Acc: 0.7078
  Epoch 10/57 - Train Loss: 0.4158, Train Acc: 0.8753, Val Loss: 0.7732, Val Acc: 0.7399
  Epoch 15/57 - Train Loss: 0.3806, Train Acc: 0.8924, Val Loss: 0.7965, Val Acc: 0.7267
  Epoch 20/57 - Train Loss: 0.3848, Train Acc: 0.8808, Val Loss: 0.8194, Val Acc: 0.7094


[I 2026-02-19 20:24:36,586] Trial 0 finished with value: -0.7399114173228346 and parameters: {'conv1_filters': 128, 'pool1_size': 3, 'sep_filters': 128, 'pool2_size': 2, 'conv3_filters': 192, 'lstm_dropout': 0.42022300234864174, 'fc_dropout': 0.5562108866694068, 'lr': 0.00010336843570697407, 'weight_decay': 0.0008123245085588687, 'epochs': 57}. Best is trial 0 with value: -0.7399114173228346.


  Early stopping at epoch 22

Trial 1
  Epoch 1/56 - Train Loss: 0.7836, Train Acc: 0.6622, Val Loss: 0.6959, Val Acc: 0.7629
  Epoch 5/56 - Train Loss: 0.4039, Train Acc: 0.8919, Val Loss: 0.8662, Val Acc: 0.6818
  Epoch 10/56 - Train Loss: 0.4028, Train Acc: 0.8857, Val Loss: 0.6967, Val Acc: 0.7782
  Epoch 15/56 - Train Loss: 0.3632, Train Acc: 0.8839, Val Loss: 0.5548, Val Acc: 0.8636
  Epoch 20/56 - Train Loss: 0.3706, Train Acc: 0.8880, Val Loss: 0.9606, Val Acc: 0.6761
  Epoch 25/56 - Train Loss: 0.3563, Train Acc: 0.9136, Val Loss: 0.5376, Val Acc: 0.8767
  Epoch 30/56 - Train Loss: 0.3493, Train Acc: 0.8841, Val Loss: 0.5305, Val Acc: 0.8818
  Epoch 35/56 - Train Loss: 0.3579, Train Acc: 0.8989, Val Loss: 0.5261, Val Acc: 0.8815
  Epoch 40/56 - Train Loss: 0.3614, Train Acc: 0.8874, Val Loss: 0.5255, Val Acc: 0.8794
  Epoch 45/56 - Train Loss: 0.3667, Train Acc: 0.8796, Val Loss: 0.5180, Val Acc: 0.8833
  Epoch 50/56 - Train Loss: 0.3566, Train Acc: 0.9129, Val Loss: 0.5589, V

[I 2026-02-19 20:50:19,971] Trial 1 finished with value: -0.8844734251968503 and parameters: {'conv1_filters': 96, 'pool1_size': 2, 'sep_filters': 192, 'pool2_size': 2, 'conv3_filters': 192, 'lstm_dropout': 0.32789877213040836, 'fc_dropout': 0.49382169728028275, 'lr': 0.00018033330377234335, 'weight_decay': 2.334586407601622e-05, 'epochs': 56}. Best is trial 1 with value: -0.8844734251968503.


...

  Epoch 15/50 - Train Loss: 0.3670, Train Acc: 0.8975, Val Loss: 0.5197, Val Acc: 0.8855
  Early stopping at epoch 15

Trial 13
  Epoch 1/47 - Train Loss: 0.7764, Train Acc: 0.6471, Val Loss: 0.7419, Val Acc: 0.7392
  Epoch 5/47 - Train Loss: 0.4065, Train Acc: 0.8915, Val Loss: 0.7325, Val Acc: 0.7709
  Epoch 10/47 - Train Loss: 0.3997, Train Acc: 0.8763, Val Loss: 0.6754, Val Acc: 0.7938
  Epoch 15/47 - Train Loss: 0.3767, Train Acc: 0.8928, Val Loss: 0.5878, Val Acc: 0.8501
  Epoch 20/47 - Train Loss: 0.3635, Train Acc: 0.8922, Val Loss: 0.5550, Val Acc: 0.8736
  Epoch 25/47 - Train Loss: 0.3558, Train Acc: 0.8940, Val Loss: 0.4967, Val Acc: 0.8987
  Epoch 30/47 - Train Loss: 0.3499, Train Acc: 0.9137, Val Loss: 0.5055, Val Acc: 0.8886
  Epoch 35/47 - Train Loss: 0.3702, Train Acc: 0.8832, Val Loss: 0.5081, Val Acc: 0.8916
  Epoch 40/47 - Train Loss: 0.3636, Train Acc: 0.8768, Val Loss: 0.5021, Val Acc: 0.8929


[I 2026-02-19 22:39:46,652] Trial 13 finished with value: -0.9030511811023622 and parameters: {'conv1_filters': 128, 'pool1_size': 2, 'sep_filters': 128, 'pool2_size': 3, 'conv3_filters': 128, 'lstm_dropout': 0.34875157562747455, 'fc_dropout': 0.5274062039738456, 'lr': 0.00022010587777443482, 'weight_decay': 4.092810888074125e-06, 'epochs': 47}. Best is trial 3 with value: -0.9036663385826772.


  Epoch 45/47 - Train Loss: 0.3533, Train Acc: 0.9066, Val Loss: 0.5051, Val Acc: 0.8923
  Early stopping at epoch 45

Trial 14
  Epoch 1/53 - Train Loss: 0.8194, Train Acc: 0.6351, Val Loss: 0.9388, Val Acc: 0.6614
  Epoch 5/53 - Train Loss: 0.4219, Train Acc: 0.8716, Val Loss: 0.8559, Val Acc: 0.7213
  Epoch 10/53 - Train Loss: 0.3731, Train Acc: 0.8801, Val Loss: 0.9631, Val Acc: 0.6873
  Epoch 15/53 - Train Loss: 0.3791, Train Acc: 0.8960, Val Loss: 0.7027, Val Acc: 0.7849
  Epoch 20/53 - Train Loss: 0.3845, Train Acc: 0.8915, Val Loss: 0.7869, Val Acc: 0.7697
  Epoch 25/53 - Train Loss: 0.3737, Train Acc: 0.8893, Val Loss: 0.5332, Val Acc: 0.8810
  Epoch 30/53 - Train Loss: 0.3659, Train Acc: 0.8947, Val Loss: 0.5190, Val Acc: 0.8865


[I 2026-02-19 22:50:43,937] Trial 14 finished with value: -0.899298720472441 and parameters: {'conv1_filters': 96, 'pool1_size': 3, 'sep_filters': 192, 'pool2_size': 3, 'conv3_filters': 128, 'lstm_dropout': 0.4090535953220017, 'fc_dropout': 0.5979900057686607, 'lr': 0.00026469580426854273, 'weight_decay': 7.504787409494396e-05, 'epochs': 53}. Best is trial 3 with value: -0.9036663385826772.


  Epoch 35/53 - Train Loss: 0.3628, Train Acc: 0.8919, Val Loss: 0.5118, Val Acc: 0.8878
  Early stopping at epoch 35

OPTIMIZATION COMPLETE
Number of finished trials: 15
Number of pruned trials: 0

Best trial:
  Best val accuracy: 0.9037

Best hyperparameters:
  conv1_filters: 96
  pool1_size: 2
  sep_filters: 192
  pool2_size: 2
  conv3_filters: 128
  lstm_dropout: 0.40401360423556215
  fc_dropout: 0.5320065419014919
  lr: 0.0001346504222274646
  weight_decay: 0.0008105016126411582
  epochs: 56


Test With TTA

In [26]:
# Load best model
best_model = ImprovedCNNLSTMModel(in_channels=8, n_classes=3, trial=study.best_trial)
best_model.to(device)
best_model.load_state_dict(study.user_attrs['best_model_state'])

# Test with TTA
all_preds, all_labels = test_with_tta(best_model, test_dl, n_tta=5)

test_acc = (all_preds == all_labels).mean()
print(f"\nTest Accuracy with TTA: {test_acc:.4f}")

# Convert to original labels
inverse_label_map = {0: 1, 1: 4, 2: 5}
all_preds_original = np.array([inverse_label_map[p] for p in all_preds])
all_labels_original = np.array([inverse_label_map[l] for l in all_labels])

print("\nClassification Report:")
print(classification_report(all_labels_original, all_preds_original,
                          target_names=['Gesture 1', 'Gesture 4', 'Gesture 5']))

print("\nConfusion Matrix:")
cm = confusion_matrix(all_labels_original, all_preds_original)
print(cm)

Testing with TTA: 100%|██████████| 127/127 [00:05<00:00, 21.25it/s]



Test Accuracy with TTA: 0.9036

Classification Report:
              precision    recall  f1-score   support

   Gesture 1       0.84      0.87      0.86      5381
   Gesture 4       0.87      0.84      0.85      5437
   Gesture 5       1.00      1.00      1.00      5438

    accuracy                           0.90     16256
   macro avg       0.90      0.90      0.90     16256
weighted avg       0.90      0.90      0.90     16256


Confusion Matrix:
[[4682  699    0]
 [ 868 4569    0]
 [   0    0 5438]]
